# Loan Data Processing (Part B1)

Assignment Five — Classification: Theory and Practice

Reproduces the Lesson 5 preprocessing pipeline (`code/loan-data-processing.py`) in notebook form, typed and run cell by cell. Step 10 substitutes a researched scaler in place of `StandardScaler`.

## 1) Load + initial snapshot

In [1]:
import pandas as pd
import numpy as np

CSV_PATH = "raw_loan_dataset.csv"
OUT_PATH = "clean_loan_dataset.csv"

df = pd.read_csv(CSV_PATH)

print("\n=== INITIAL HEAD ===")
print(df.head())



=== INITIAL HEAD ===
    Income  CreditScore  ...  PreviousDefaults  Approved
0   108810        537.0  ...                No        No
1    96482        524.0  ...                No       Yes
2    28478          NaN  ...                No       Yes
3  $25,851        561.0  ...                No       Yes
4    38396        527.0  ...                No  Approved

[5 rows x 7 columns]


In [2]:
print("\n=== INITIAL INFO ===")
print(df.info())



=== INITIAL INFO ===
<class 'pandas.DataFrame'>
RangeIndex: 103 entries, 0 to 102
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Income            98 non-null     str    
 1   CreditScore       97 non-null     float64
 2   EmploymentYears   98 non-null     float64
 3   LoanAmount        98 non-null     str    
 4   HasCollateral     101 non-null    str    
 5   PreviousDefaults  101 non-null    str    
 6   Approved          103 non-null    str    
dtypes: float64(2), str(5)
memory usage: 5.8 KB
None


In [3]:
print("\n=== INITIAL MISSING VALUES ===")
print(df.isnull().sum())



=== INITIAL MISSING VALUES ===
Income              5
CreditScore         6
EmploymentYears     5
LoanAmount          5
HasCollateral       2
PreviousDefaults    2
Approved            0
dtype: int64


## 2) Clean currency formatting

In [4]:
df["Income"] = df["Income"].replace(r"[\$,]", "", regex=True).astype(float)
df["LoanAmount"] = df["LoanAmount"].replace(r"[\$,]", "", regex=True).astype(float)

print(df[["Income", "LoanAmount"]].dtypes)


Income        float64
LoanAmount    float64
dtype: object


## 3) Normalize categorical typos BEFORE imputation

In [5]:
yes_no_map = {
    "yes": "Yes", "y": "Yes", "yse": "Yes", "1": "Yes", "approved": "Yes",
    "no": "No", "n": "No", "0": "No", "rejected": "No",
}

df["HasCollateral"] = df["HasCollateral"].astype(str).str.strip().str.lower().replace(yes_no_map).replace({"nan": np.nan})
df["PreviousDefaults"] = df["PreviousDefaults"].astype(str).str.strip().str.lower().replace(yes_no_map).replace({"nan": np.nan})
df["Approved"] = df["Approved"].astype(str).str.strip().str.lower().replace(yes_no_map).replace({"nan": np.nan})

print(df["HasCollateral"].value_counts(dropna=False))
print(df["PreviousDefaults"].value_counts(dropna=False))
print(df["Approved"].value_counts(dropna=False))


HasCollateral
No     51
Yes    50
NaN     2
Name: count, dtype: int64
PreviousDefaults
No     86
Yes    15
NaN     2
Name: count, dtype: int64
Approved
Yes    68
No     35
Name: count, dtype: int64


## 4) Impute missing values

In [6]:
df["Income"] = df["Income"].fillna(df["Income"].median())
df["CreditScore"] = df["CreditScore"].fillna(df["CreditScore"].median())
df["EmploymentYears"] = df["EmploymentYears"].fillna(df["EmploymentYears"].median())
df["LoanAmount"] = df["LoanAmount"].fillna(df["LoanAmount"].median())
df["HasCollateral"] = df["HasCollateral"].fillna(df["HasCollateral"].mode()[0])
df["PreviousDefaults"] = df["PreviousDefaults"].fillna(df["PreviousDefaults"].mode()[0])

print(df.isnull().sum())


Income              0
CreditScore         0
EmploymentYears     0
LoanAmount          0
HasCollateral       0
PreviousDefaults    0
Approved            0
dtype: int64


## 5) Remove duplicates

In [7]:
before = df.shape[0]
df = df.drop_duplicates()
print(f"\nDropped duplicates: {before} -> {df.shape[0]} rows")



Dropped duplicates: 103 -> 100 rows


## 6) IQR capping on numeric columns

Clip extreme values to the IQR fence (k=1.5) instead of deleting rows.

In [8]:
def iqr_fun(series, k=1.5):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower = q1 - k * iqr
    upper = q3 + k * iqr
    return lower, upper

low_income,  high_income  = iqr_fun(df["Income"])
low_credit,  high_credit  = iqr_fun(df["CreditScore"])
low_loan,    high_loan    = iqr_fun(df["LoanAmount"])
low_emp,     high_emp     = iqr_fun(df["EmploymentYears"])

df["Income"]          = df["Income"].clip(lower=low_income,  upper=high_income)
df["CreditScore"]     = df["CreditScore"].clip(lower=low_credit, upper=high_credit)
df["LoanAmount"]      = df["LoanAmount"].clip(lower=low_loan,    upper=high_loan)
df["EmploymentYears"] = df["EmploymentYears"].clip(lower=low_emp,     upper=high_emp)

df[["Income", "CreditScore", "LoanAmount", "EmploymentYears"]].describe()


## 7) Label encoding (Yes/No -> 0/1)

Approved=1, Rejected=0

In [9]:
df["Approved"] = df["Approved"].map({"Yes": 1, "No": 0}).astype(int)

print("\n=== CLASS DISTRIBUTION (after label encoding) ===")
print(df["Approved"].value_counts())
print(df["Approved"].value_counts(normalize=True).round(3))



=== CLASS DISTRIBUTION (after label encoding) ===
Approved
1    66
0    34
Name: count, dtype: int64
Approved
1    0.66
0    0.34
Name: proportion, dtype: float64


## 8) Class balance check

Imbalanced classes can make Accuracy misleading.

In [10]:
class_ratio = df["Approved"].value_counts(normalize=True).min()
if class_ratio < 0.30:
    print("\nWarning: severely imbalanced classes -- consider balancing techniques.")
else:
    print("\nClass balance OK for baseline Accuracy (both classes well represented).")

df["HasCollateral"] = df["HasCollateral"].map({"Yes": 1, "No": 0}).astype(int)
df["PreviousDefaults"] = df["PreviousDefaults"].map({"Yes": 1, "No": 0}).astype(int)



Class balance OK for baseline Accuracy (both classes well represented).


## 9) Feature engineering (no leakage from label)

In [11]:
df["DebtToIncome"] = df["LoanAmount"] / df["Income"].replace(0, np.nan)
df["IncomePerYearEmployed"] = df["Income"] / (df["EmploymentYears"] + 1)

df[["DebtToIncome", "IncomePerYearEmployed"]].describe()


## 10) Feature scaling — researched scaler (not StandardScaler)

**Chosen scaler: `RobustScaler` (from `sklearn.preprocessing`)**

The lesson script uses `StandardScaler`, which centers on the mean and scales by standard deviation — both sensitive to outliers. `RobustScaler` centers on the **median** and scales by the **interquartile range (IQR)** instead, so it is much less affected by extreme values.

This fits the loan dataset because `Income`, `LoanAmount`, and the engineered `DebtToIncome` ratio are naturally skewed, even after IQR capping in step 6. A median/IQR-based scaler keeps typical applicants well-spread in the transformed space instead of letting a few remaining large values dominate the scale.

Excludes the label (`Approved`) and the already-binary columns from scaling, same as the lesson script.

**Source:** scikit-learn documentation, *RobustScaler* — https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.RobustScaler.html

In [12]:
from sklearn.preprocessing import RobustScaler

binary_cols = {"HasCollateral", "PreviousDefaults", "Approved"}
numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
scale_cols = [c for c in numeric_cols if c not in binary_cols]

scaler = RobustScaler()
df[scale_cols] = scaler.fit_transform(df[scale_cols])

df[scale_cols].head()


## 11) Final snapshot + save

Features (X) = all columns except `Approved`; label (y) = `Approved`.

In [13]:
print("\n=== FINAL HEAD ===")
print(df.head())



=== FINAL HEAD ===
     Income  CreditScore  ...  DebtToIncome  IncomePerYearEmployed
0  0.822149    -0.653722  ...     -0.445244               8.274536
1  0.564574    -0.737864  ...     -0.912749               0.153994
2 -0.856268     0.000000  ...      0.280113              -0.126321
3 -0.911156    -0.498382  ...     -0.315175              -0.669033
4 -0.649046    -0.718447  ...     -0.284688              -0.284975

[5 rows x 9 columns]


In [14]:
print("\n=== FINAL INFO ===")
print(df.info())



=== FINAL INFO ===
<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Income                 100 non-null    float64
 1   CreditScore            100 non-null    float64
 2   EmploymentYears        100 non-null    float64
 3   LoanAmount             100 non-null    float64
 4   HasCollateral          100 non-null    int64  
 5   PreviousDefaults       100 non-null    int64  
 6   Approved               100 non-null    int64  
 7   DebtToIncome           100 non-null    float64
 8   IncomePerYearEmployed  100 non-null    float64
dtypes: float64(6), int64(3)
memory usage: 7.2 KB
None


In [15]:
print("\n=== FINAL MISSING VALUES ===")
print(df.isnull().sum())



=== FINAL MISSING VALUES ===
Income                   0
CreditScore              0
EmploymentYears          0
LoanAmount               0
HasCollateral            0
PreviousDefaults         0
Approved                 0
DebtToIncome             0
IncomePerYearEmployed    0
dtype: int64


In [16]:
df.to_csv(OUT_PATH, index=False)
print(f"\nSaved cleaned dataset to {OUT_PATH}")



Saved cleaned dataset to clean_loan_dataset.csv
